In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns

import random
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import StackingClassifier

import warnings
warnings.simplefilter(action='ignore',category=FutureWarning)

In [2]:
data = pd.read_csv("storm_data.csv")
data.head()

,Date,Time,Type,Wind,Pressure,Latitude(°N),Longitude(°E),Humidity(%),Temperature(°C),Visibility(km),Precipitation(mm),SST(°C),Storm Surge(m),Wave Height(m),Air Density(kg/m³)
0,12-May,18:00,Severe Cyclonic Storm,132,991,18.7,82.0,81,22,0,50,14,0.3,1.7,1.225
1,12-May,18:00,Cyclonic Storm,164,938,14.3,78.6,78,9,1,2,15,0.7,0.6,1.225
2,12-May,6:00,Disturbance,151,992,12.2,82.5,86,23,8,85,11,0.7,0.6,1.225
3,12-May,0:00,Cyclonic Storm,47,999,17.5,78.2,97,4,0,56,20,0.1,0.7,1.225
4,13-May,12:00,Severe Cyclonic Storm,130,956,8.7,84.7,88,20,6,9,13,0.5,0.9,1.225


In [3]:
data=data.drop(["Time"],axis = 1)
data=data.drop(["Date"],axis = 1)

In [4]:
data.describe()

,Wind,Pressure,Latitude(°N),Longitude(°E),Humidity(%),Temperature(°C),Visibility(km),Precipitation(mm),SST(°C),Storm Surge(m),Wave Height(m),Air Density(kg/m³)
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000
mean,126.567000,970.410000,14.148100,78.873800,87.098000,15.412000,6.827000,49.886000,18.271000,0.818100,1.050200,1.225
std,53.986721,20.357955,3.414252,3.460929,7.214877,6.849611,4.408458,28.511996,5.201982,0.405372,0.549091,0.000
min,35.000000,935.000000,8.000000,73.000000,75.000000,4.000000,0.000000,1.000000,10.000000,0.100000,0.100000,1.225
25%,76.750000,952.000000,11.300000,75.900000,81.000000,10.000000,3.000000,27.000000,14.000000,0.500000,0.600000,1.225
50%,129.000000,971.000000,14.150000,78.800000,87.000000,15.000000,7.000000,50.000000,18.000000,0.800000,1.100000,1.225
75%,174.250000,989.000000,17.200000,81.900000,93.000000,21.000000,11.000000,74.000000,23.000000,1.200000,1.500000,1.225
max,219.000000,1004.000000,20.000000,85.000000,99.000000,27.000000,14.000000,99.000000,27.000000,1.500000,2.000000,1.225


In [5]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

data["Type"] = le.fit_transform(data["Type"])

class_mapping = dict(zip(le.classes_, le.transform(le.classes_)))

print("Class Mapping:", class_mapping)

Class Mapping: {'Cyclonic Storm': 0, 'Depression': 1, 'Disturbance': 2, 'Extremely Severe Cyclonic Storm': 3, 'Low Pressure Area': 4, 'Severe Cyclonic Storm': 5, 'Very Severe Cyclonic Storm': 6}


In [6]:
features = [feat for feat in data.columns if feat !='Type']

X = data[features] # feature set
y = data['Type'] # target


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1, stratify=y)

X_train.shape, X_test.shape

((800, 12), (200, 12))

In [7]:
print(f"The dataset contains {data.shape[0]} rows and {data.shape[1]} columns")

num_features = [feat for feat in features if data[feat].dtype != object]
cat_features = [feat for feat in features if data[feat].dtype == object]

print(f"Total number of features : {len(features)}")
print(f"Number of numerical features : {len(num_features)}")
print(f"Number of categorical features : {len(cat_features)}\n")

The dataset contains 1000 rows and 13 columns
Total number of features : 12
Number of numerical features : 12
Number of categorical features : 0



In [8]:
scaler = MinMaxScaler(feature_range=(0, 1))

X_train[num_features] = scaler.fit_transform(X_train[num_features]) #fit and transform the train set
X_test[num_features] = scaler.transform(X_test[num_features]) #transform the test test


In [9]:
X_train.drop(['Latitude(°N)'], axis=1, inplace=True)
X_test.drop(['Latitude(°N)'], axis=1, inplace=True)

In [10]:
X_train.drop(['Longitude(°E)'], axis=1, inplace=True)
X_test.drop(['Longitude(°E)'], axis=1, inplace=True)

In [11]:
tree = DecisionTreeClassifier(random_state=1)
tree.fit(X_train, y_train)

print("Train accuracy : ", accuracy_score(y_train, tree.predict(X_train)))
print("Test accuracy : ", accuracy_score(y_test, tree.predict(X_test)))

Train accuracy :  1.0
Test accuracy :  0.13


In [12]:
# 🎯 Define hyperparameters
distribution = {'max_depth': [4, 6, 8, 10, 12, 14, 16],
                'criterion': ['gini', 'entropy'],
                'min_samples_split': [2, 10, 20, 30, 40],
                'max_features': [0.2, 0.4, 0.6, 0.8, 1],
                'max_leaf_nodes': [8, 16, 32, 64, 128, 256],
                'class_weight': [{0: 1, 1: 2}, {0: 1, 1: 3}, {0: 1, 1: 4}, {0: 1, 1: 5}]
               }

# 🎯 Random search for best hyperparameters
search = RandomizedSearchCV(DecisionTreeClassifier(random_state=1),
                         distribution,
                         scoring='accuracy',
                         cv=3,
                         verbose=1,
                         random_state=1,
                         n_iter=30)

# 🎯 Fit the randomized search
search.fit(X_train, y_train)

# 🎯 Best parameters for Decision Tree classifier
search.best_params_


Fitting 3 folds for each of 30 candidates, totalling 90 fits


{'min_samples_split': 40,
 'max_leaf_nodes': 64,
 'max_features': 0.4,
 'max_depth': 16,
 'criterion': 'gini',
 'class_weight': {0: 1, 1: 3}}

In [13]:
# Retrain with best model

best_tree = search.best_estimator_

best_tree.fit(X_train, y_train)
print(" Best train accuracy : ", accuracy_score(y_train, best_tree.predict(X_train)))
print(" Best test accuracy : ", accuracy_score(y_test, best_tree.predict(X_test)))

 Best train accuracy :  0.31125
 Best test accuracy :  0.145


In [14]:
# Hyperparameters
params_grid = {'bootstrap': [True, False],
             'max_depth': [2, 5, 10, 20, None],
             'max_features': ['auto', 'sqrt'],
             'min_samples_leaf': [1, 2, 4],
             'min_samples_split': [2, 5, 10],
             'n_estimators': [50, 100, 150, 200]}

# Random search for best hyperparameters
search = RandomizedSearchCV(RandomForestClassifier(random_state=1),
                         params_grid,
                         scoring='accuracy',
                         cv=3,
                         verbose=1,
                         random_state=1,
                         n_iter=20)

search.fit(X_train, y_train)

# Best parameters for Random forest classifier
search.best_params_

Fitting 3 folds for each of 20 candidates, totalling 60 fits


c:\Users\rudra\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py:547: FitFailedWarning: 
27 fits failed out of a total of 60.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
27 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\rudra\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\rudra\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py", line 1467, in wrapper
    estimator._validate_params()
  File "c:\Users\rudra\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py

{'n_estimators': 150,
 'min_samples_split': 5,
 'min_samples_leaf': 1,
 'max_features': 'sqrt',
 'max_depth': 5,
 'bootstrap': False}

In [15]:
# Retrain with best model

best_forest = search.best_estimator_

best_forest.fit(X_train, y_train)
print("Best train accuracy : ", accuracy_score(y_train, best_forest.predict(X_train)))
print("Best test accuracy : ", accuracy_score(y_test, best_forest.predict(X_test)))

Best train accuracy :  0.58875
Best test accuracy :  0.12


In [16]:
import pickle
with open('DIC_final.pkl', 'wb') as file:
    pickle.dump(best_forest, file)
